In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage, ToolMessage
from dotenv import load_dotenv
import os
from langgraph.checkpoint.memory import MemorySaver,InMemorySaver
load_dotenv()

True

In [2]:
# LLM Configuration - DeepSeek R1 Distill Qwen 7B
LLAMA_STUDIO_API_BASE = os.getenv("LLAMA_STUDIO_API_BASE", "http://localhost:1234/v1")
LLAMA_STUDIO_API_KEY = os.getenv("LLAMA_STUDIO_API_KEY", "lm-studio")
LLM_MODEL_NAME = os.getenv("LLM_MODEL_NAME", "deepseek-r1-distill-qwen-7b")
LLM_TEMPERATURE = float(os.getenv("LLM_TEMPERATURE", "0.7"))
LLM_MAX_TOKENS = int(os.getenv("LLM_MAX_TOKENS", "2048"))
LLM_TIMEOUT = int(os.getenv("LLM_TIMEOUT", "60"))  # Timeout in seconds
LLM_MAX_RETRIES = int(os.getenv("LLM_MAX_RETRIES", "2"))  # Number of retries


class ChatBot(TypedDict):
    #Reducer function add message used here 
    message: Annotated[list[BaseMessage], add_messages]


llm = ChatOpenAI(
    base_url=LLAMA_STUDIO_API_BASE,
    api_key=LLAMA_STUDIO_API_KEY,
    model=LLM_MODEL_NAME,
    temperature=LLM_TEMPERATURE,
    max_tokens=LLM_MAX_TOKENS,
    timeout=LLM_TIMEOUT,
    max_retries=LLM_MAX_RETRIES,
)


def chat_node(state: ChatBot):
    messsage = state["message"]
    res = llm.invoke(messsage)
    return {"message": res}

In [3]:
ch = MemorySaver()
graph = StateGraph(ChatBot)
graph.add_node("chat_node", chat_node)
graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chat_bot = graph.compile(checkpointer=ch)

In [4]:
# in_state = {"message": [HumanMessage(content="What is Capital oF Pakistan")]}
# chat_bot.invoke(in_state)

In [5]:
thread_id = '1'
while True:
    user_msg = input("type Here")
    print("User", user_msg)
    if user_msg.strip().lower() in ["exit", "quit", "bye"]:
        break
    config = {'configurable':{'thread_id':thread_id}}
    response = chat_bot.invoke({"message": [HumanMessage(content=user_msg)]},config=config)
    print("AI", response["message"][-1].content)

User what is my name
AI I don't have access to personal information, so I wouldn't be able to know or tell you your name. How can I assist you otherwise?
User exit


In [7]:
chat_bot.get_state(config=config)

StateSnapshot(values={'message': [HumanMessage(content='what is my name', additional_kwargs={}, response_metadata={}, id='c98151a0-f443-45bc-a971-f2be29c60047'), AIMessage(content="I don't have access to personal information, so I wouldn't be able to know or tell you your name. How can I assist you otherwise?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 23, 'total_tokens': 53, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen2.5-7b-instruct-1m', 'system_fingerprint': 'qwen2.5-7b-instruct-1m', 'id': 'chatcmpl-vva4bkdj28wc1jc38df6g', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--ec4b6502-25ff-49ab-89e9-72ecef0c2d5f-0', usage_metadata={'input_tokens': 23, 'output_tokens': 30, 'total_tokens': 53, 'input_token_details': {}, 'output_token_details': {}})]}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f

In [9]:
### Persistance 
